In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler

# Load dataset

In [2]:
file_path = "./data/house_prices.csv"
df = pd.read_csv(file_path)
print("First 5 rows:")
print(df.head())
print("\nColumns:")
print(df.columns.tolist())

First 5 rows:
   Unnamed: 0  Elevator  Parking  Year  Area (decimeter^2)   Cost
0           0         1        0  1379               20000   5900
1           1         0        0  1371               10000   2900
2           2         0        1  1379               10000   2900
3           3         1        0  1385               40000  11900
4           4         1        0  1379               20000   5900

Columns:
['Unnamed: 0', 'Elevator', 'Parking', 'Year', 'Area (decimeter^2)', 'Cost']


# Define features and target

In [3]:
feature_cols = ["Elevator", "Parking", "Year", "Area (decimeter^2)"]
target_col = "Cost"
X = df[feature_cols].copy()
y = df[target_col].copy()
print(X.isna().sum())
# Fill missing values if any
X = X.fillna(X.mean())
y = y.fillna(y.mean())

Elevator              0
Parking               0
Year                  0
Area (decimeter^2)    0
dtype: int64


# Evaluation function

In [ ]:
"""
Train a decision tree regressor, evaluate its performance using R2, RMSE, and MAE, and rank input features based on their importance.
"""
def eval_tree_regressor(X, y, title='Model'):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = DecisionTreeRegressor(random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Useful info
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    importances = pd.Series(
        model.feature_importances_,
        index = X.columns
    ).sort_values(ascending=False)

    print("\n" + "=" * 50)
    print(title)
    print("=" * 50)
    print("Feature importances:")
    print(importances)
    print(f"R2   : {r2:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"MAE  : {mae:.4f}")

    return {
        "model": model,
        "importances": importances,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae
    }

# Full experiment

In [5]:
def run_experiment(X, y, experiment_name='Without Normalization'):
    print("\n" + "#" * 60)
    print(f"Experiment: {experiment_name}")
    print("#" * 60)

    # Model 1: all features
    result_all = eval_tree_regressor(X, y, title="Model 1: All Features")
    top_feature = result_all["importances"].idxmax()
    X_top = X[[top_feature]]

    # Model 2: only the most important feature
    result_top = eval_tree_regressor(
        X_top, y, title=f"Model 2: Only Most Important Feature ({top_feature})"
    )

    summary = pd.DataFrame([
        {
            "Experiment": experiment_name,
            "Model": "All Features",
            "Used Features": ", ".join(X.columns),
            "R2": result_all["R2"],
            "RMSE": result_all["RMSE"],
            "MAE": result_all["MAE"]
        },
        {
            "Experiment": experiment_name,
            "Model": "Only Most Important Feature",
            "Used Features": top_feature,
            "R2": result_top["R2"],
            "RMSE": result_top["RMSE"],
            "MAE": result_top["MAE"]
        }
    ])

    return summary, result_all["importances"], top_feature

# Part 1: Without normalization

In [6]:
summary_without_norm, importances_without_norm, top_feature_without_norm = run_experiment(
    X, y, experiment_name="Without Normalization"
)


############################################################
Experiment: Without Normalization
############################################################

Model 1: All Features
Feature importances:
Area (decimeter^2)    0.999625
Year                  0.000321
Parking               0.000031
Elevator              0.000023
dtype: float64
R2   : 0.9996
RMSE : 35.3553
MAE  : 12.5000

Model 2: Only Most Important Feature (Area (decimeter^2))
Feature importances:
Area (decimeter^2)    1.0
dtype: float64
R2   : 0.9986
RMSE : 63.4379
MAE  : 42.2100


# Part 2: With normalization

In [7]:
scaler = MinMaxScaler()
X_norm = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns
)
summary_with_norm, importances_with_norm, top_feature_with_norm = run_experiment(
    X_norm, y, experiment_name="With Normalization"
)


############################################################
Experiment: With Normalization
############################################################

Model 1: All Features
Feature importances:
Area (decimeter^2)    0.999625
Year                  0.000321
Parking               0.000031
Elevator              0.000023
dtype: float64
R2   : 0.9996
RMSE : 35.3553
MAE  : 12.5000

Model 2: Only Most Important Feature (Area (decimeter^2))
Feature importances:
Area (decimeter^2)    1.0
dtype: float64
R2   : 0.9986
RMSE : 63.4379
MAE  : 42.2100


In [8]:
X_norm.head()

,Elevator,Parking,Year,Area (decimeter^2)
0,1.0,0.0,0.591837,0.2
1,0.0,0.0,0.428571,0.0
2,0.0,1.0,0.591837,0.0
3,1.0,0.0,0.714286,0.6
4,1.0,0.0,0.591837,0.2


# Final comparison

In [9]:
final_summary = pd.concat(
    [summary_without_norm, summary_with_norm],
    ignore_index=True
)
final_summary

,Experiment,Model,Used Features,R2,RMSE,MAE
0,Without Normalization,All Features,"Elevator, Parking, Year, Area (decimeter^2)",0.999572,35.355339,12.500000
1,Without Normalization,Only Most Important Feature,Area (decimeter^2),0.998622,63.437935,42.210012
2,With Normalization,All Features,"Elevator, Parking, Year, Area (decimeter^2)",0.999572,35.355339,12.500000
3,With Normalization,Only Most Important Feature,Area (decimeter^2),0.998622,63.437935,42.210012


In [10]:
print("\nMost important feature without normalization:", top_feature_without_norm)
print("Most important feature with normalization   :", top_feature_with_norm)


Most important feature without normalization: Area (decimeter^2)
Most important feature with normalization   : Area (decimeter^2)
